In [12]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from pathlib import Path
import random

SEED = 42

# 재현성 함수 정의
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def capture_rng_state():
    numpy_state = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": {
            "bit_generator": numpy_state[0],
            "state": numpy_state[1].tolist(),
            "pos": int(numpy_state[2]),
            "has_gauss": int(numpy_state[3]),
            "cached_gaussian": float(numpy_state[4]),
        },
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        )
    }

def restore_rng_state(rng_state):
    random.setstate(rng_state["python"])
    numpy_state = rng_state["numpy"]
    np.random.set_state((
        numpy_state["bit_generator"],
        np.asarray(numpy_state["state"], dtype=np.uint32),
        int(numpy_state["pos"]),
        int(numpy_state["has_gauss"]),
        float(numpy_state["cached_gaussian"])
    ))
    torch.set_rng_state(rng_state["torch_cpu"].cpu())
    if torch.cuda.is_available() and rng_state["torch_cuda"] is not None:
        torch.cuda.set_rng_state_all([
            state.cpu() for state in rng_state["torch_cuda"]
        ])

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [13]:
# 데이터 준비

num_samples = 240
X = torch.randn(num_samples, 4)
true_W = torch.tensor([
    [1.0, -1.0, 0.5],
    [0.5, 1.5, -1.0],
    [-1.0, 0.5, 1.0],
    [-1.0, 0.2, -0.5],
])
y = (X @ true_W).argmax(dim=1)

dataset = TensorDataset(X, y)

split_generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(dataset, [192, 48], generator=split_generator)

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, generator=loader_generator)
val_loader = DataLoader(val_dataset, batch_size=48, shuffle=False)

In [14]:
# 모델 및 학습, 검증 함수 정의

class SimpleMLP(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=32, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        logits = self.fc2(x)
        return logits

def train_one_epoch(model, train_loader, loss_fn, optimizer, device):
    model.train()
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_samples = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_x)
        loss = loss_fn(logits, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * batch_y.size(0)
        epoch_correct += (logits.argmax(dim=1) == batch_y).sum().item()
        epoch_samples += batch_y.size(0)

    avg_loss = epoch_loss / epoch_samples
    avg_acc = epoch_correct / epoch_samples

    return avg_loss, avg_acc

def evaluate(model, val_loader, loss_fn, device):
    model.eval()
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_samples = 0

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_x)
            loss = loss_fn(logits, batch_y)

            epoch_loss += loss.item() * batch_y.size(0)
            epoch_correct += (logits.argmax(dim=1) == batch_y).sum().item()
            epoch_samples += batch_y.size(0)

    avg_loss = epoch_loss / epoch_samples
    avg_acc = epoch_correct / epoch_samples

    return avg_loss, avg_acc

In [15]:
def save_checkpoint(model, optimizer, completed_epochs, history, config, loader_generator, path):
    if any(len(values) != completed_epochs for values in history.values()):
        raise ValueError("모든 history 길이는 completed_epochs와 같아야 합니다.")

    checkpoint = {
        "completed_epochs": int(completed_epochs),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_valid_loss": float(min(history["val_loss"])),
        "history": history,
        "config": config,
        "rng_state": capture_rng_state(),
        "loader_generator_state": loader_generator.get_state(),
        "early_stopping_state_dict": None,
    }

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, temporary_path)
    temporary_path.replace(path)

In [16]:
config = {
    "model_name": "SimpleMLP",
    "input_dim": 4,
    "hidden_dim": 32,
    "num_classes": 3,
    "learning_rate": 0.001,
    "batch_size": 16,
    "seed": SEED,
}

model = SimpleMLP(
    input_dim = config["input_dim"],
    hidden_dim = config["hidden_dim"],
    num_classes = config["num_classes"],
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"])

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

for epoch in range(1, 3):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"epoch={epoch} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"vak_acc={val_acc:.4f} | "
    )

checkpoint_path = Path(r"C:\Users\limux\Documents\1. Projects\KANT Private LLM Infra Engineer\kant-llm-engineer-main\step1_llm-foundations\ch03_deep-learning-basic\checkpoints\last.pt")
save_checkpoint(
    model = model,
    optimizer = optimizer,
    completed_epochs = 2,
    history = history,
    config = config,
    loader_generator = loader_generator,
    path = checkpoint_path,
)
print("2 epoch checkpoint 저장:", checkpoint_path)

epoch=1 | train_loss=1.0719 | train_acc=0.3698 | val_loss=0.9904 | vak_acc=0.5833 | 
epoch=2 | train_loss=1.0232 | train_acc=0.5521 | val_loss=0.9399 | vak_acc=0.7083 | 
2 epoch checkpoint 저장: C:\Users\limux\Documents\1. Projects\KANT Private LLM Infra Engineer\kant-llm-engineer-main\step1_llm-foundations\ch03_deep-learning-basic\checkpoints\last.pt


In [17]:
# 모델, optimizer, train DataLoader용 generator 생성

resumed_model = SimpleMLP(
    input_dim = config["input_dim"],
    hidden_dim = config["hidden_dim"],
    num_classes = config["num_classes"]
).to(device)

resumed_optimizer = torch.optim.Adam(
    resumed_model.parameters(),
    lr = config["learning_rate"]
)

resumed_loader_generator = torch.Generator()
resumed_train_loader = DataLoader(
    train_dataset,
    batch_size = config["batch_size"],
    shuffle = True,
    generator = resumed_loader_generator,
)

checkpoint = torch.load(
    checkpoint_path,
    map_location = device,
    weights_only = True,
)
print(checkpoint.keys())
# dict_keys(['completed_epochs', 'model_state_dict', 'optimizer_state_dict', 'best_valid_loss', 'history', 'config', 'rng_state', 'loader_generator_state', 'early_stopping_state_dict'])

dict_keys(['completed_epochs', 'model_state_dict', 'optimizer_state_dict', 'best_valid_loss', 'history', 'config', 'rng_state', 'loader_generator_state', 'early_stopping_state_dict'])


In [18]:
# 모델과 optimizer 복원, RNG와 generator 상태 복원

if checkpoint["config"] != config:
    raise ValueError("checkpoint config가 현재 설정과 다릅니다.")

resumed_model.load_state_dict(checkpoint["model_state_dict"])
resumed_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

restore_rng_state(checkpoint["rng_state"])
resumed_loader_generator.set_state(checkpoint["loader_generator_state"].cpu())

completed_epochs = int(checkpoint["completed_epochs"])
start_epoch = completed_epochs + 1
history = checkpoint["history"]

if any(len(values) != completed_epochs for values in history.values()):
    raise ValueError("checkpoint history 길이가 completed_epochs와 다릅니다.")

model = resumed_model
optimizer = resumed_optimizer
train_loader = resumed_train_loader

print(f"완료된 epoch: {completed_epochs}")
print(f"다시 시작할 epoch: {start_epoch}")

완료된 epoch: 2
다시 시작할 epoch: 3


In [19]:
# start_epoch 부터 학습 재개하기
# 매 epoch가 끝날 때 checkpoint를 갱신

total_epochs = 5

for epoch in range(start_epoch, total_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    save_checkpoint(
        model = model,
        optimizer = optimizer,
        completed_epochs = epoch,
        history = history,
        config = config,
        loader_generator = resumed_loader_generator,
        path = checkpoint_path,
    )

    print(
        f"epoch={epoch} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"valid_loss={val_loss:.4f} | "
        f"valid_acc={val_acc:.4f}"
    )

epoch=3 | train_loss=0.9774 | train_acc=0.6406 | valid_loss=0.8933 | valid_acc=0.7500
epoch=4 | train_loss=0.9361 | train_acc=0.6771 | valid_loss=0.8502 | valid_acc=0.7917
epoch=5 | train_loss=0.8952 | train_acc=0.7396 | valid_loss=0.8083 | valid_acc=0.8542


In [21]:
# 저장 결과 검증

verified = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

assert int(verified["completed_epochs"]) == 5
assert all(len(values) == 5 for values in verified["history"].values())

print(f"최종 완료 epoch: {verified["completed_epochs"]}")
print("history 길이:", {
    key: len(values)
    for key, values in verified["history"].items()
    })

최종 완료 epoch: 5
history 길이: {'train_loss': 5, 'train_acc': 5, 'val_loss': 5, 'val_acc': 5}


In [ ]:
# 평가나 추론

inference_model = SimpleMLP(
    input_dim = config["input_dim"],
    hidden_dim = config["hidden_dim"],
    num_classes = config["num_classes"],
).to(device)

inference_checkpoint = torch.load(
    checkpoint_path,
    map_loaction = device,
    weights_only = True,
)
inference_model.load_state_dict(
    inference_checkpoint["model_state_dict"]
)
inference_model.eval()

with torch.inference_mode():
    sample_logits = inference_model(X[:5].to(device))
    sample_predictions = sample_logits.argmax(dim=1)

print("예측 class:", sample_predictions.cpu())